In [1]:
from importlib import import_module
from pathlib import Path
import json
import random
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / '.data/extracted').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

evaluation = import_module('src.steps.03_evaluation')
EvaluationItem, Evaluator = evaluation.EvaluationItem, evaluation.Evaluator
SAMPLE_SIZE, SEED, WORKERS = 15, 42, 2

In [2]:
def load_items(limit=1, seed=42):
    extracted = PROJECT_ROOT / '.data/extracted'
    annotated = PROJECT_ROOT / '.data/annotated'
    texts = {p.relative_to(extracted).with_suffix('').as_posix(): p for p in extracted.rglob('*.txt')}
    labels = {p.relative_to(annotated).with_suffix('').as_posix(): p for p in annotated.rglob('*.json')}
    pairs = sorted(texts.keys() & labels.keys())
    if not pairs:
        raise FileNotFoundError('No matching resume text/JSON pairs found.')
    if limit is not None:
        pairs = random.Random(seed).sample(pairs, min(limit, len(pairs)))

    items = []
    for pair in pairs:
        target = json.loads(labels[pair].read_text(encoding='utf-8'))
        items.append(EvaluationItem(
            resume_id=Path(pair).stem,
            category=Path(pair).parent.as_posix(),
            resume_text=texts[pair].read_text(encoding='utf-8'),
            target_json=target,
        ))
    return items, len(texts.keys() & labels.keys())

In [3]:
items, available_pairs = load_items(SAMPLE_SIZE, SEED)
print(f'Available paired resumes: {available_pairs:,}')
print(f'Loaded for this test: {len(items)}')

Available paired resumes: 13,591
Loaded for this test: 15


In [4]:
def generate_response(item):
    return item.target_json  

In [6]:
results = Evaluator(items).evaluate(
    callback=generate_response,
    workers=3,
)

Evaluating with Jev (3 workers): 100%|██████████| 15/15 [00:03<00:00,  4.45resume/s]


In [ ]:
for result in results.results:
    print(result.judge_score * 100)

100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0


## Evaluate your model

Replace `your_model.generate(...)` with your inference call, then run this cell. Set `SAMPLE_SIZE` above to the number of pairs you want to test.

In [ ]:
def model_callback(item):
    prompt = evaluation.render_json_prompt(item.resume_text)
    return your_model.generate(prompt)

# report = Evaluator(items).evaluate(model_callback, workers=WORKERS)
# print(json.dumps(report.metrics, indent=2))